<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing L2 data (CALLIPSO)**

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> Run Notebook in binder from NASA-tutorial Github repository, or
3. <font size="3"> Get `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
- <font size="3"> Inspecting metadata.
- <font size="3"> Subset by variable names.
-  <font size="3"> Subset by spatial selection.
- <font size="3"> Streaming data into local file


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

In [ ]:
Callipso_L2_ccid = "C3463063995-LARC_CLOUD" # 
time_range = [dt.datetime(2023, 5, 15), dt.datetime(2023, 5, 20)] # One month of data

cmr_urls = get_cmr_urls(ccid=Callipso_L2_ccid, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[0]

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()

# Accessing Metadata

<font size="3.5"> What are some tools, their differences, and what do they do.

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **PYDAP: Metadata-Only**



```{note}
Q: How do we tell the server which protocol to use?
A: By replaing the http -> dap4 in the URL
```


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
output_path = "data/"

keep_variables = [
    '/Lidar_Surface_Detection', "/Ocean_Derived_Column_Optical_Depth",
    "/Lidar_Data_Altitudes", "/Profile_ID", "/Latitude", "/Longitude", 
    "/Profile_Time", "/Profile_UTC_Time", "/Day_Night_Flag", "/Tropopause_Height", 
    "/Tropopause_Temperature","/Snow_Ice_Surface_Type", "/Opacity_Flag", 
]

### Stream data

In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, keep_variables=keep_variables, output_path=output_path)

## check the data

In [ ]:
dt = xr.open_datatree(output_path+"CAL_LID_L2_01kmCLay-Standard-V5-00.2023-05-17T15-41-01ZN.nc4")
dt